<div style="background: linear-gradient(135deg, #1f3b57 0%, #2f6f8f 100%); color: white; padding: 28px 32px; border-radius: 10px;">
<h1 style="margin: 0; color: white;">XGBoost from Scratch</h1>
<p style="margin: 8px 0 0 0; font-size: 1.1em; opacity: 0.9;">Deriving and implementing gradient-boosted trees with second-order optimisation, using only NumPy</p>
<p style="margin: 14px 0 0 0; font-size: 0.9em; opacity: 0.75;">Carles Trullas</p>
</div>

This notebook builds a minimal but faithful version of **XGBoost** from first principles and benchmarks it against the official `xgboost` library on two synthetic problems: a regression task and a binary classification task.

The goal is understanding, not speed: every piece of the algorithm (the regularised objective, the optimal leaf weights, the split gain and the boosting loop) is written explicitly so the mathematics maps one-to-one onto the code.

### Contents

1. [How XGBoost works](#1.-How-XGBoost-works)
2. [Setup](#2.-Setup)
3. [Building blocks: nodes and trees](#3.-Building-blocks:-nodes-and-trees)
4. [The boosting ensemble](#4.-The-boosting-ensemble)
5. [Regression experiment](#5.-Regression-experiment)
6. [Classification experiment](#6.-Classification-experiment)
7. [Takeaways](#7.-Takeaways)

## 1. How XGBoost works

XGBoost is an additive model: the prediction for a sample $x_i$ after $t$ boosting rounds is a sum of regression trees,

$$
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \, f_t(x_i),
$$

where $\eta$ is the learning rate and $f_t$ is the tree added at round $t$. Each new tree is chosen to minimise a **regularised objective**:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} \ell\!\left(y_i,\; \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \gamma T + \tfrac{1}{2}\lambda \sum_{j=1}^{T} w_j^2,
$$

with $T$ the number of leaves, $w_j$ the leaf weights, $\gamma$ a penalty per leaf and $\lambda$ an L2 penalty on leaf weights.

### Second-order approximation

The key idea of XGBoost is to approximate the loss with a second-order Taylor expansion around the current prediction. Defining the gradient and hessian of each sample,

$$
g_i = \frac{\partial \ell(y_i, \hat{y}_i)}{\partial \hat{y}_i}, \qquad h_i = \frac{\partial^2 \ell(y_i, \hat{y}_i)}{\partial \hat{y}_i^2},
$$

and grouping samples by the leaf $I_j$ they fall into, with $G_j = \sum_{i \in I_j} g_i$ and $H_j = \sum_{i \in I_j} h_i$, the objective becomes a simple quadratic in each $w_j$.

### Optimal leaf weight

Minimising that quadratic gives the weight of every leaf in closed form:

$$
w_j^{*} = -\frac{G_j}{H_j + \lambda}.
$$

### Split gain

Plugging $w_j^*$ back in, the quality of splitting a node into a left part $L$ and a right part $R$ is

$$
\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda}\right].
$$

A node is only split when the best gain exceeds $\gamma$, which acts as built-in pruning.

### Loss functions used here

| Task | Loss $\ell(y, \hat{y})$ | Gradient $g$ | Hessian $h$ | Initial prediction |
|---|---|---|---|---|
| Regression | $(y - \hat{y})^2$ | $2(\hat{y} - y)$ | $2$ | $\bar{y}$ |
| Binary classification | log loss on $p = \sigma(\hat{y})$ | $p - y$ | $p(1 - p)$ | $\log\frac{\bar{y}}{1 - \bar{y}}$ |

For classification, $\hat{y}$ lives in log-odds space and is mapped to a probability with the sigmoid $\sigma$.

## 2. Setup

The only dependency beyond the usual scientific stack is the official `xgboost` package, used purely as a reference to compare against.

In [ ]:
# Uncomment if xgboost is not installed in your environment
# %pip install xgboost

In [ ]:
import time
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

CUSTOM_COLOR = "#2f6f8f"
NATIVE_COLOR = "#e07a3f"

## 3. Building blocks: nodes and trees

### The `Node` class

A node receives the samples that reach it together with their gradients $g$ and hessians $h$, and decides whether to become a leaf or to split. Its job maps directly onto the formulas above:

- `leaf_weight` implements $w^* = -G / (H + \lambda)$.
- `split_gain` implements the gain formula.
- `find_best_split` scans every feature. After sorting the samples by feature value, cumulative sums of $g$ and $h$ give $G_L, H_L$ for **every** candidate split at once, so the scan is vectorised instead of a Python loop. Positions where two consecutive samples share the same value are skipped, because no threshold can separate them.
- A node becomes a leaf when it holds a single sample, reaches `max_depth`, has no valid split, or when the best gain is below $\gamma$.

In [ ]:
class Node:
    # A single node of an XGBoost regression tree.

    def __init__(self):
        self.is_leaf: bool = False
        self.left: Optional["Node"] = None
        self.right: Optional["Node"] = None
        self.feature_idx: Optional[int] = None
        self.threshold: Optional[float] = None
        self.weight: Optional[float] = None

    # ------------------------------------------------------------------ math
    @staticmethod
    def leaf_weight(G, H, lambda_):
        # Optimal leaf weight: w* = -G / (H + lambda)
        return -G / (H + lambda_)

    @staticmethod
    def score(G, H, lambda_):
        # Structure score of a leaf: G^2 / (H + lambda)
        return G ** 2 / (H + lambda_)

    def split_gain(self, G_left, H_left, G_right, H_right, lambda_):
        # Reduction in the objective obtained by splitting a node in two
        G, H = G_left + G_right, H_left + H_right
        return 0.5 * (self.score(G_left, H_left, lambda_)
                      + self.score(G_right, H_right, lambda_)
                      - self.score(G, H, lambda_))

    # --------------------------------------------------------------- fitting
    def fit(self, X, g, h, depth, max_depth, lambda_, gamma):
        if len(X) == 1 or depth >= max_depth:
            self._make_leaf(g, h, lambda_)
            return

        best_gain, best_split = self.find_best_split(X, g, h, lambda_)

        # No valid split, or not worth the gamma penalty: stop growing
        if best_split is None or best_gain < gamma:
            self._make_leaf(g, h, lambda_)
            return

        self.feature_idx, self.threshold, left_idx, right_idx = best_split

        self.left = Node()
        self.left.fit(X[left_idx], g[left_idx], h[left_idx],
                      depth + 1, max_depth, lambda_, gamma)

        self.right = Node()
        self.right.fit(X[right_idx], g[right_idx], h[right_idx],
                       depth + 1, max_depth, lambda_, gamma)

    def _make_leaf(self, g, h, lambda_):
        self.is_leaf = True
        self.weight = self.leaf_weight(np.sum(g), np.sum(h), lambda_)

    def find_best_split(self, X, g, h, lambda_):
        G, H = np.sum(g), np.sum(h)
        best_gain, best_split = -np.inf, None

        for feature_idx in range(X.shape[1]):
            order = np.argsort(X[:, feature_idx], kind="stable")
            values = X[order, feature_idx]

            # Left-side statistics for every split position "after sample k"
            G_left = np.cumsum(g[order])[:-1]
            H_left = np.cumsum(h[order])[:-1]
            G_right, H_right = G - G_left, H - H_left

            gains = self.split_gain(G_left, H_left, G_right, H_right, lambda_)

            # A threshold can only separate samples with different values
            gains[values[:-1] == values[1:]] = -np.inf
            if gains.size == 0:
                continue

            k = int(np.argmax(gains))
            if gains[k] > best_gain:
                best_gain = gains[k]
                best_split = (feature_idx, values[k], order[:k + 1], order[k + 1:])

        return best_gain, best_split

    # ------------------------------------------------------------ prediction
    def predict(self, X):
        # Vectorised prediction for a 2D array of samples
        if self.is_leaf:
            return np.full(X.shape[0], self.weight)

        out = np.empty(X.shape[0])
        goes_left = X[:, self.feature_idx] <= self.threshold
        out[goes_left] = self.left.predict(X[goes_left])
        out[~goes_left] = self.right.predict(X[~goes_left])
        return out

### The `Tree` class

A thin wrapper that owns the root node and exposes `fit` and `predict`.

In [ ]:
class Tree:
    # A regression tree fitted on gradients and hessians.

    def __init__(self, max_depth, lambda_, gamma):
        self.max_depth = max_depth
        self.lambda_ = lambda_
        self.gamma = gamma
        self.root = Node()

    def fit(self, X, g, h):
        self.root.fit(X, g, h, depth=0, max_depth=self.max_depth,
                      lambda_=self.lambda_, gamma=self.gamma)
        return self

    def predict(self, X):
        return self.root.predict(X)

## 4. The boosting ensemble

Regression and classification share exactly the same boosting loop; they only differ in the loss, which enters through three methods: the initial prediction, the gradient and the hessian. That is captured with a small base class and two subclasses.

The training loop:

1. Start every sample at the same initial prediction (the mean target, or the log-odds of the positive class).
2. At each round, compute $g$ and $h$ at the current predictions.
3. Fit a tree on $(X, g, h)$.
4. Update the running predictions with $\eta \cdot f_t(X)$.

Keeping the running predictions in memory means each round only evaluates the newly added tree, instead of re-running the whole ensemble.

In [ ]:
class BaseCustomXGBoost:
    # Shared boosting logic. Subclasses define the loss.

    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.3,
                 lambda_=1.0, gamma=0.0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.lambda_ = lambda_
        self.gamma = gamma
        self.estimators: List[Tree] = []
        self.base_score: float = 0.0

    # Loss-specific hooks --------------------------------------------------
    def initial_prediction(self, y):
        raise NotImplementedError

    def gradient(self, y, raw):
        raise NotImplementedError

    def hessian(self, y, raw):
        raise NotImplementedError

    # Boosting loop --------------------------------------------------------
    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.estimators = []
        self.base_score = self.initial_prediction(y)
        raw = np.full(len(y), self.base_score)

        for _ in range(self.n_estimators):
            g = self.gradient(y, raw)
            h = self.hessian(y, raw)

            tree = Tree(self.max_depth, self.lambda_, self.gamma).fit(X, g, h)
            self.estimators.append(tree)

            raw += self.learning_rate * tree.predict(X)
        return self

    def raw_predict(self, X):
        # Ensemble output before any link function (value or log-odds)
        X = np.asarray(X, dtype=float)
        raw = np.full(X.shape[0], self.base_score)
        for tree in self.estimators:
            raw += self.learning_rate * tree.predict(X)
        return raw

    def staged_raw_predict(self, X):
        # Yield the raw prediction after each boosting round
        X = np.asarray(X, dtype=float)
        raw = np.full(X.shape[0], self.base_score)
        for tree in self.estimators:
            raw = raw + self.learning_rate * tree.predict(X)
            yield raw

### Regressor

Squared error loss: $g = 2(\hat{y} - y)$ and a constant hessian $h = 2$.

In [ ]:
class CustomXGBoostRegressor(BaseCustomXGBoost):

    def initial_prediction(self, y):
        return float(np.mean(y))

    def gradient(self, y, raw):
        return 2.0 * (raw - y)

    def hessian(self, y, raw):
        return np.full(len(y), 2.0)

    def predict(self, X):
        return self.raw_predict(X)

    def score(self, X, y):
        return r2_score(y, self.predict(X))

### Classifier

Log loss with the model working in log-odds space: $g = p - y$ and $h = p(1-p)$, where $p = \sigma(\hat{y})$.

In [ ]:
class CustomXGBoostClassifier(BaseCustomXGBoost):

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def initial_prediction(self, y):
        p = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        return float(np.log(p / (1 - p)))

    def gradient(self, y, raw):
        return self.sigmoid(raw) - y

    def hessian(self, y, raw):
        p = self.sigmoid(raw)
        return p * (1 - p)

    def predict_proba(self, X):
        p = self.sigmoid(self.raw_predict(X))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

### Evaluation helper

Trains a model, times it, and returns the train and test scores so both implementations can be compared side by side.

In [ ]:
def evaluate(name, model, X_train, y_train, X_test, y_test, metric_name):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    return {
        "Model": name,
        "Training time (s)": round(elapsed, 3),
        f"Train {metric_name}": round(model.score(X_train, y_train), 4),
        f"Test {metric_name}": round(model.score(X_test, y_test), 4),
    }

## 5. Regression experiment

A synthetic regression problem from `make_regression`, with an 80/20 train/test split. Both models use the same hyperparameters: 100 trees, depth 3, learning rate 0.3, $\lambda = 1$ and $\gamma = 0$. These match the defaults of the `xgboost` library.

In [ ]:
X, y = make_regression(random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
params = dict(n_estimators=100, max_depth=3, learning_rate=0.3)

custom_reg = CustomXGBoostRegressor(**params, lambda_=1.0, gamma=0.0)
native_reg = xgboost.XGBRegressor(**params, reg_lambda=1.0, gamma=0.0,
                                  tree_method="exact", random_state=4)

reg_results = pd.DataFrame([
    evaluate("Custom XGBoost", custom_reg, X_train, y_train, X_test, y_test, "R²"),
    evaluate("Native XGBoost", native_reg, X_train, y_train, X_test, y_test, "R²"),
]).set_index("Model")

reg_results

### Visual comparison

On the left, predicted against true values on the test set (points on the diagonal are perfect predictions). On the right, how test $R^2$ evolves as trees are added to the custom model, with the final score of the native model as a reference.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# Predicted vs true
lims = [min(y_test.min(), custom_reg.predict(X_test).min()),
        max(y_test.max(), custom_reg.predict(X_test).max())]
ax1.plot(lims, lims, color="grey", lw=1, ls="--", label="Perfect prediction")
ax1.scatter(y_test, custom_reg.predict(X_test), s=40, alpha=0.8,
            color=CUSTOM_COLOR, label="Custom")
ax1.scatter(y_test, native_reg.predict(X_test), s=40, alpha=0.8, marker="^",
            color=NATIVE_COLOR, label="Native")
ax1.set(xlabel="True value", ylabel="Predicted value", title="Test set: predicted vs true")
ax1.legend(frameon=False)

# Test R² per boosting round
staged_r2 = [r2_score(y_test, raw) for raw in custom_reg.staged_raw_predict(X_test)]
ax2.plot(range(1, len(staged_r2) + 1), staged_r2, color=CUSTOM_COLOR, lw=2, label="Custom")
ax2.axhline(native_reg.score(X_test, y_test), color=NATIVE_COLOR, ls="--", lw=1.5,
            label="Native (final)")
ax2.set(xlabel="Number of trees", ylabel="Test R²", title="Learning curve")
ax2.legend(frameon=False)

plt.tight_layout()
plt.show()

## 6. Classification experiment

A deliberately hard binary problem: `make_classification` with `class_sep=0.1`, so the two classes overlap heavily. Same hyperparameters as before, scored with accuracy.

In [ ]:
X, y = make_classification(n_samples=1000, class_sep=0.1, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}  |  Positive rate: {y.mean():.2f}")

In [ ]:
custom_clf = CustomXGBoostClassifier(**params, lambda_=1.0, gamma=0.0)
native_clf = xgboost.XGBClassifier(**params, reg_lambda=1.0, gamma=0.0,
                                   tree_method="exact", random_state=3)

clf_results = pd.DataFrame([
    evaluate("Custom XGBoost", custom_clf, X_train, y_train, X_test, y_test, "accuracy"),
    evaluate("Native XGBoost", native_clf, X_train, y_train, X_test, y_test, "accuracy"),
]).set_index("Model")

clf_results

### Visual comparison

On the left, the predicted probability of the positive class from both models on the test set; points close to the diagonal mean both implementations agree. On the right, train and test accuracy per boosting round for the custom model, which shows how quickly the ensemble starts to overfit on such a noisy problem.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# Agreement between implementations
p_custom = custom_clf.predict_proba(X_test)[:, 1]
p_native = native_clf.predict_proba(X_test)[:, 1]
ax1.plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
ax1.scatter(p_native, p_custom, c=y_test, cmap="coolwarm", s=30, alpha=0.8, edgecolor="none")
ax1.set(xlabel="Native P(y = 1)", ylabel="Custom P(y = 1)",
        title=f"Probability agreement (corr = {np.corrcoef(p_custom, p_native)[0, 1]:.3f})")

# Accuracy per boosting round
sig = CustomXGBoostClassifier.sigmoid
train_acc = [accuracy_score(y_train, sig(r) > 0.5) for r in custom_clf.staged_raw_predict(X_train)]
test_acc = [accuracy_score(y_test, sig(r) > 0.5) for r in custom_clf.staged_raw_predict(X_test)]
rounds = range(1, len(test_acc) + 1)
ax2.plot(rounds, train_acc, color=CUSTOM_COLOR, lw=2, alpha=0.5, label="Custom (train)")
ax2.plot(rounds, test_acc, color=CUSTOM_COLOR, lw=2, label="Custom (test)")
ax2.axhline(native_clf.score(X_test, y_test), color=NATIVE_COLOR, ls="--", lw=1.5,
            label="Native test (final)")
ax2.set(xlabel="Number of trees", ylabel="Accuracy", title="Learning curve")
ax2.legend(frameon=False)

plt.tight_layout()
plt.show()

## 7. Takeaways

- **The whole algorithm fits in a few formulas.** A second-order Taylor expansion of the loss turns each boosting round into a problem with a closed-form solution: $w^* = -G/(H+\lambda)$ for the leaves and a gain expression for the splits.
- **The loss is pluggable.** Regression and classification share the exact same tree-building and boosting code; only the gradient, the hessian and the initial prediction change. This is what lets XGBoost support arbitrary differentiable objectives.
- **Regularisation is built into the tree structure.** $\lambda$ shrinks leaf weights towards zero and $\gamma$ prunes splits that do not reduce the objective enough, rather than relying only on post-hoc pruning.
- **The custom model should track the native one closely**, since both use the exact greedy split search with the same hyperparameters. Remaining differences come from implementation details (for instance, how ties and split thresholds are handled).
- **What the native library adds on top** is mostly engineering: histogram-based and approximate split finding, sparsity-aware handling of missing values, column and row subsampling, parallelism, and cache-aware memory layout. This is where its large speed advantage comes from.